### GD_Landsat_02_Download selects Landsat images of SEAN glaciers and downloads them.
DONE, but: multiple years/multiple satellites/whole record  
TODO: figure out scaling of above.  
See various geemap materials - Courses and Notebooks, as well as Google Developers for Google Earth Engine  
https://geemap.org/notebooks/01_geemap_intro/  
https://developers.google.com/earth-engine/guides/image_overview  
See also: GD_Landsat_01_Setup.ipynb, test_proj, test_roi, gee_intro/AssetManagement/export_data, image_visualization, 50_cartoee_quickstart, geemap_AKB, test_landsat_geemap  
Test outputs are here: C:\Users\andyb\Documents\U\GEE-Courses\data and shown in test_geemap.qgz  

https://www.usgs.gov/faqs/what-are-band-designations-landsat-satellites

set up for Landsat in UTM8N. True for path 059. Path 060 uses UTM7N (grr!).  
Note that each image band's metadata has crs info e.g. 'crs': 'EPSG:32608', 'crs_transform': [30, 0, 289785, 0, -30, 6631515]}  

In [ ]:
import ee
import geemap
import pandas as pd
import os
from shapely.geometry import box, Polygon
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px

In [ ]:
#this or ee.Initialize
Map = geemap.Map()

In [ ]:
folder_base = r'C:\Users\andyb\Documents\U\SEAN_Glacier-Dynamics' #os.path.join()
#folder_shp = r'C:\Users\andyb\Documents\U\GEE-Courses\data' #get away from this...
folder_land = r'C:\Users\andyb\Documents\U\GlacierLandsat'
file_path=os.path.join(folder_base,'glacierPropsLandsat.csv')

glaciers = pd.read_csv(file_path) #contains Name, LatCenter, LonCenter, two types of bounding boxes (see GD_Landsat_01_Setup).
glaciers[['Name','Region']]

In [ ]:
#choose one glacier 
glacier = glaciers.iloc[44]
glacierdf=glaciers.iloc[[44]]
glacierName_Region = glacier['Name'] + '_' + glacier['Region']
print('You chose: ' + glacierName_Region)
folder_out=os.path.join(folder_land, glacierName_Region)
#folder_fig=os.path.join(folder_out, 'Figures') #now just use folder_out
folder_img=os.path.join(folder_out, 'Images')
os.makedirs(folder_out, exist_ok=True)
#os.makedirs(folder_fig, exist_ok=True)
os.makedirs(folder_img, exist_ok=True)

In [ ]:
glacierPt = ee.Geometry.Point(glacier['LonCenter'],glacier['LatCenter']) #-137.121, 58.838) #Johns Hopkins (center of terminus)
#TODO: explore polygon instead (may work seamlessly, may not)
#SEE ALSO: reducing/clipping to area in "create an image composite" section of reducing_image_collection.ipynb
Map.centerObject(glacierPt, 12)  # Zoom level 12 for a close view
# Add a marker at the point (optional, for visualization)
Map.addLayer(glacierPt, {'color': 'red'}, glacier['Name'])

In [ ]:
#add region of interest
#roi = ee.Geometry.Rectangle([glacier['LonMin'], glacier['LatMin'], glacier['LonMax'], glacier['LatMax']])
def load_polygon(df):
    coords = []
    i = 1
    while f'x{i}' in df.columns and f'y{i}' in df.columns:
        x = df[f'x{i}'].iloc[0]
        y = df[f'y{i}'].iloc[0]
        coords.append((x, y))
        i += 1
    # Ensure the polygon is closed (first and last points are the same)
    if coords and coords[0] != coords[-1]:
        coords.append(coords[0])
    return Polygon(coords)

roi=load_polygon(glacierdf)
#shapely back to ee_polygon: Get the exterior coordinates as a list of lists
roicoords = [list(roi.exterior.coords)]
roi_ee = ee.Geometry.Polygon(roicoords)
print(roi)
#print(roi_ee)

In [ ]:
# Create a feature with the region name
feature = ee.Feature(roi_ee, {'name': glacier['Name']+" "+glacier['Region']})
# Add the region to the map with a unique color or style
Map.addLayer(feature, {'color': 'red'}, glacier['Name']+" "+glacier['Region'])
Map

In [ ]:
#2014 Landsat 8. 
#see GD_MORW_2024_GEEDiT2_04 for examples of other satellites ~line 400
#collection = (ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA').filterDate('2014-01-01', '2015-01-01').filterBounds(glacierPt).sort('system:time_start'))
#collectionSR = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2014-01-01', '2015-01-01').filterBounds(glacierPt).sort('system:time_start'))
#print(collection.size().getInfo())

#collection.aggregate_array("system:id").getInfo()

#CONCLUSION: top of atmosphere has 3 extra images (25 total) compared to surface reflectance (22 total):  
 #'LANDSAT/LC08/C02/T1_TOA/LC08_059019_20141224', #extra
 #'LANDSAT/LC08/C02/T1_TOA/LC08_060019_20141113', #extra
 #'LANDSAT/LC08/C02/T1_TOA/LC08_060019_20141129'] #extra
#100% Cloudy? NO, but low sun angle...

In [ ]:
#Landsat 4,5,7,8,9 all years.
#was T1_L2 for all, changed to T1_TOA for better dynamic range with snow. 
landsat4 = ee.ImageCollection('LANDSAT/LT04/C02/T1_TOA').filterBounds(glacierPt) # Landsat 4 TM (1982-1993) 27602 total everywhere. 2 for Margerie
landsat5 = ee.ImageCollection('LANDSAT/LT05/C02/T1_TOA').filterBounds(glacierPt) # Landsat 5 TM (1984-2013) 1905311. 295
landsat7 = ee.ImageCollection('LANDSAT/LE07/C02/T1_TOA').filterBounds(glacierPt) # Landsat 7 ETM+ (1999-present) 2608926. 437
landsat8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA').filterBounds(glacierPt) # Landsat 8 (2013-present) 2035359 and counting. 294
landsat9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_TOA').filterBounds(glacierPt) # Landsat 9 (2021-present) 653482 and counting. 76

#ee.ImageCollection('LANDSAT/LM01/C02/T1_L2')) to the merge chain (note: coarser 60m res, different band mapping).

In [ ]:
collection = landsat4.merge(landsat5).merge(landsat7).merge(landsat8).merge(landsat9) \
    .filterBounds(glacierPt) \
    .sort('system:time_start') \
    .filterDate('1980-01-01', '2025-10-1') #actual begin is probably 1982 
#total 1103 images for Margerie

In [ ]:
print(landsat4.size().getInfo()) #Margerie
print(landsat5.size().getInfo()) #
print(landsat7.size().getInfo()) #
print(landsat8.size().getInfo()) #
print(landsat9.size().getInfo()) #
print(f'Total {collection.size().getInfo()}') #~30 for L8 1 year, ~300 for L8 all years, ~1000 for all sats all years
#collection.aggregate_array("system:id").getInfo()

In [ ]:
#collection[1100].getInfo()
#DONE: go back to small subset (L8) and try scale factors, then L4.

#collection

collection.first()

In [ ]:
# Apply scaling factors (Landsat C2 SR is stored as DN*0.000027 + -0.2)
#def apply_scale_factors(image):
#    optical = image.select('SR_B.').multiply(0.000027).add(-0.2)
#    thermal = image.select('ST_B10').multiply(0.00341802).add(149.0)
#    return image.addBands(optical, None, True) \
#                .addBands(thermal, None, True)

#collection_scaled = collectionSR.map(apply_scale_factors) #was landsat9
#collection_scaled
#collection_scaled.first()

In [ ]:
# Fetch metadata for all images in the collection and create a metadata DataFrame with the info
mdf = pd.DataFrame({
    'system:id': collection.aggregate_array('system:id').getInfo(), # Image IDs (with path)
    'system:index': collection.aggregate_array('system:index').getInfo(), # Image IDs (just image name)
    'DATE_ACQUIRED': collection.aggregate_array('DATE_ACQUIRED').getInfo(), # date acquired
    'system:time_start': collection.aggregate_array('system:time_start').getInfo(), # Acquisition timestamps
    'CLOUD_COVER': collection.aggregate_array('CLOUD_COVER').getInfo(),  # Cloud cover %
    'CLOUD_COVER_LAND': collection.aggregate_array('CLOUD_COVER_LAND').getInfo()  # Cloud cover land %
    #anything else I should keep?
})

In [ ]:
#mdfs = pd.DataFrame({
#    'system:id': collection_scaled.aggregate_array('system:id').getInfo(), # Image IDs (with path)
#    'system:index': collection_scaled.aggregate_array('system:index').getInfo(), # Image IDs (just image name)
#    'DATE_ACQUIRED': collection_scaled.aggregate_array('DATE_ACQUIRED').getInfo(), # date acquired
#    'system:time_start': collection_scaled.aggregate_array('system:time_start').getInfo(), # Acquisition timestamps
#    'CLOUD_COVER': collection_scaled.aggregate_array('CLOUD_COVER').getInfo(),  # Cloud cover %
#    'CLOUD_COVER_LAND': collection_scaled.aggregate_array('CLOUD_COVER_LAND').getInfo()  # Cloud cover land %
#    #anything else I should keep?
#})

In [ ]:
# Convert timestamps to readable dates (double check that acquired date is same as image start time). Guessing this is UTC.
mdf['datetime'] = pd.to_datetime(mdf['system:time_start'], unit='ms')
mdf['datetimestr'] = pd.to_datetime(mdf['system:time_start'], unit='ms').dt.strftime('%Y-%m-%d %H:%M:%S')

In [ ]:
#mdf.head()
#mdf.tail()

In [ ]:
# Save metadata to LandsatMetadata CSV
info_file = os.path.join(folder_out, glacierName_Region + '_' + 'LandsatMetadata.csv')
mdf.to_csv(info_file, index=False)
print(f"Image info saved to: {info_file}")

In [ ]:
#type(mdf) #pandas.core.frame.DataFrame
#print(mdf.columns) #dtypes de facto lists the columns
print(mdf.dtypes)

#type(mdf['datetime'][0]) #pandas._libs.tslibs.timestamps.Timestamp

In [ ]:
# Count points per year
mdf['year'] = mdf['datetime'].dt.year
mdf['DayOfYear'] = mdf['datetime'].dt.dayofyear
points_per_year = mdf['year'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 6))
# Bar chart
bars = ax.bar(points_per_year.index, points_per_year.values,
              color='steelblue', edgecolor='black', alpha=0.8)
# Add value labels on top of bars
#for bar in bars:
#    height = bar.get_height()
#    ax.text(bar.get_x() + bar.get_width()/2, height + 0.5,
#            f'{int(height)}', ha='center', va='bottom', fontsize=10)

ax.set_xlabel('Year')
ax.set_ylabel('Images')
ax.grid(True, axis='y', linestyle='--', alpha=0.5)
ax.set_xticks(points_per_year.index) # Optional: force integer x-ticks
ax.tick_params(axis='x', rotation=90) #better(?) than plt.xticks(rotation=90)
ax.text(0.03, 0.95, glacierName_Region,transform=ax.transAxes,ha='left',va='top',fontsize=12,color='black')
fig.savefig(os.path.join(folder_out,glacierName_Region+'_CountPerYear.png'), dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
#Plot cloud cover over time
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(mdf['datetime'], mdf['CLOUD_COVER_LAND'], marker='o', markersize=3, linewidth=0)
# Let Matplotlib choose the best locator/formatter automatically
fig.autofmt_xdate()                     # slant labels, avoid overlap
# Or, if you want full control:
# from matplotlib.dates import AutoDateLocator, ConciseDateFormatter
# locator = AutoDateLocator(minticks=5, maxticks=10)
# formatter = ConciseDateFormatter(locator)
# ax.xaxis.set_major_locator(locator)
# ax.xaxis.set_major_formatter(formatter)
ax.set_xlabel('Date')
ax.set_ylabel('Cloud cover over land')
#ax.set_title('Datetime vs Value (auto-formatted labels)')
ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
ax.text(0.03, 0.95, glacierName_Region,transform=ax.transAxes,ha='left',va='top',fontsize=12,color='black')
fig.savefig(os.path.join(folder_out,glacierName_Region+'_CloudCoverTimeSeries.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#Plot cloud cover vs dayofyear
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(mdf['DayOfYear'], mdf['CLOUD_COVER_LAND'], marker='o', markersize=3, linewidth=0)
# Let Matplotlib choose the best locator/formatter automatically
fig.autofmt_xdate()                     # slant labels, avoid overlap
# Or, if you want full control:
# from matplotlib.dates import AutoDateLocator, ConciseDateFormatter
# locator = AutoDateLocator(minticks=5, maxticks=10)
# formatter = ConciseDateFormatter(locator)
# ax.xaxis.set_major_locator(locator)
# ax.xaxis.set_major_formatter(formatter)
ax.set_xlabel('Day of Year')
ax.set_ylabel('Cloud cover over land')
#ax.set_title('Datetime vs Value (auto-formatted labels)')
ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
ax.text(0.03, 0.95, glacierName_Region,transform=ax.transAxes,ha='left',va='top',fontsize=12,color='black')
fig.savefig(os.path.join(folder_out,glacierName_Region+'_CloudCoverOverYear.png'), dpi=300, bbox_inches='tight')
plt.show()

#Climatology example 
#TODO: convert to use cloud cover data (and other things)
# 1. Create dummy 20-year data (replacing with your CSV load if needed)
dates = pd.date_range(start='2004-01-01', end='2023-12-31', freq='D')
temp_values = 15 - 10 * np.cos(2 * np.pi * dates.dayofyear / 365.25) + np.random.normal(0, 3, len(dates))
df = pd.DataFrame({'Date': dates, 'Temp': temp_values})

# 2. Extract day of year and year
df['DayOfYear'] = df['Date'].dt.dayofyear
df['Year'] = df['Date'].dt.year

# 3. Calculate 20-year average and extremes for each day
climatology = df.groupby('DayOfYear')['Temp'].agg(['mean', 'min', 'max'])

# 4. Plotting
plt.figure(figsize=(12, 6))

# Plot the 20-year mean
plt.plot(climatology.index, climatology['mean'], label='20-Year Average', color='red', linewidth=2)

# Optional: Fill between min and max to show the historical range
plt.fill_between(climatology.index, climatology['min'], climatology['max'], color='gray', alpha=0.2, label='Temp Range (20 yrs)')

plt.title('20-Year Temperature Climatology by Day of Year')
plt.xlabel('Day of Year (1-366)')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()


In [ ]:
#fig = px.line(mdf, x='datetime', y='CLOUD_COVER_LAND', title='Datetime vs Cloud cover over land')
##fig.update_xaxes(tickformat='%b %d\n%Y', tickangle=0)
#fig.show()

In [ ]:
#Plot cloud cover vs cloud cover land
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(mdf['CLOUD_COVER'], mdf['CLOUD_COVER_LAND'], marker='o', markersize=3, lw=0)
ax.set_xlabel('Cloud cover')
ax.set_ylabel('Cloud cover over land')
#ax.set_title('Datetime vs Value (auto-formatted labels)')
ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
ax.text(0.03, 0.95, glacierName_Region,transform=ax.transAxes,ha='left',va='top',fontsize=12,color='black')
fig.savefig(os.path.join(folder_out,glacierName_Region+'_CloudCoverLandVsTotal.png'), dpi=300, bbox_inches='tight')
plt.show()
#CONCLUSION: it can be cloudy over water and not over land, or cloudy over land and not water in Glacier Bay.

In [ ]:
# Set visualization parameters
#USE FOR SURFACE REFLECTANCE
vis_paramsSR = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'],'min': 2000,'max': 150000} # #was 8000, was 18000
#USE FOR SURFACE REFLECTANCE that has been scaled.
vis_trueScaled = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'],'min': 0.05,'max': 1.6}
#USE FOR SURFACE REFLECTANCE that has been scaled.
vis_falseScaled = {'bands': ['SR_B5', 'SR_B4', 'SR_B3'],'min': 0.05,'max': 1.6}
#TOP OF ATMOSPHERE TRUE
vis_params = {'bands': ['B4', 'B3', 'B2'],'min': 0.05,'max': 1.6} #was 0.3 or 0. for normal land scenes, not snow
#for Landsat 4 and 5, GEEDiT uses bands:['B3','B2','B1'],gamma:1.5,min:0,max:0.8
vis_params45 = {'bands': ['B3', 'B2', 'B1'],'min': 0.05,'max': 1.6}
#example in test_landsat_geemap used SR, here we use TOA, so delete the SR_ prefix.
vis_true = {'bands': ['B4', 'B3', 'B2'],'min': 0.0,'max': 0.3,'gamma': 1.4}
vis_false = {'bands': ['B5', 'B4', 'B3'],'min': 0.0,'max': 0.3,'gamma': 1.4} # NIR, Red, Green

#SEE ALSO: test_vis_params.ipynb and C:\Users\andyb\Documents\U\GEE-Courses\data\landsat_vis_exports
#Landsat 8 or 9 with gamma, top of atmosphere
vis_params = {"bands": ['B4', 'B3', 'B2'], "min": 0, "max": 1.5, "gamma":1.4} #AKB best of all
#Landsat 4,5,7 with gamma
vis_params45 = {"bands": ['B3', 'B2', 'B1'], "min": 0, "max": 1.5, "gamma":1.4} #AKB best of all

#Add the Landsat image to the map (visualize with true color bands)
#Map.addLayer(collection_scaled.first(), vis_trueScaled, "ScaleFirst "+mdfs['DATE_ACQUIRED'][0])
#Map.addLayer(collection_scaled.first(), vis_falseScaled, "ScaleFirst false "+mdfs['DATE_ACQUIRED'][0])
Map.addLayer(collection.filterDate('2021-12-01', '2021-12-5'),vis_params, "vis_params ~2021-12-1") #just added, untested.
Map.addLayer(collection.first(), vis_params, "First params"+mdf['DATE_ACQUIRED'][0])
Map.addLayer(collection.first(), vis_false, "First false "+mdf['DATE_ACQUIRED'][0])
#Map.addLayer(first_imageSR, vis_paramsSR, "First image SR "+first_date)
Map
#OLD CONCLUSION: SR product seems to have the highlights washed out - details seen in TOA not present in SR, even with max limit set quite high. 
#Would be nice to be more systematic about this, but enough for now...
#Is there a way to plot the histogram of values? Or set min/max to 5%/95%?

## Time Series of Images

In [ ]:
#Map.add_time_slider(collection, vis_params, labels=mdf['DATE_ACQUIRED'], time_interval=1)
#ERROR: ImageCollection.toBands: Too many bands (more than 5000).
#date_start=pd.Timestamp('1980-1-1')
#date_end=pd.Timestamp('1989-12-31')
date_start=pd.Timestamp('2025-1-1')
date_end=pd.Timestamp('2025-12-31')
#mdfi= #indexed to date
mdff = mdf.loc[(mdf['datetime'] >= date_start) & (mdf['datetime'] <= date_end)].reset_index(drop=True)

mdff['DATE_ACQUIRED']

Map.add_time_slider(collection.filterDate(date_start, date_end), vis_params, labels=mdff['DATE_ACQUIRED'], time_interval=1)
Map

## Download Collection - one at a time
DONE: Where in the download is there reprojection? Nowhere. Google Earth is in WGS84. Landsat single scenes are in UTM8N or 7N for this part of the world. Some Landsat examples (e.g. SF airport) are larger mosaics so they also use WGS84. GEEDiT for glaciers shows the same as what I see here.  
Johns Hopkins ones come out as EPSG:32608 UTM8N.  
Other examples come out as WGS84... Proj4: +proj=utm +zone=8 +datum=WGS84 +units=m +no_defs  

DONE (see test_vis_params): Standardize/understand display parameters in jupyter and in QGIS.  
NOTE: parameters vary based on Lansdat satellite.  

In [ ]:
# Function to clip each image in the collection to the ROI
def clip_image(image):
    #return image.clip(roi)
    return image.clip(roi_ee)

# Apply clipping to the entire collection
collection_clip = collection.map(clip_image)

#collection_clip.first()

In [ ]:
print(type(collection_clip.first()))
print(collection_clip.first().get('system:id').getInfo())
print(collection_clip.first().get('DATE_ACQUIRED').getInfo())
print(type(collection_clip.first().get('system:id').getInfo()))
#print(collection_clip.first().get('system:id'))
print(type(collection_clip.first().get('system:id')))
first_date = collection_clip.first().get('DATE_ACQUIRED').getInfo()

In [ ]:
Map.addLayer(collection_clip.first(), vis_params, "First clipped "+first_date)
#Map.addLayer(collection_clip.first(), vis_params, "First clipped ")

In [ ]:
#export .tif
#geemap.ee_export_image_collection(collection_clip, out_dir=folder_out)

In [ ]:
#Convert collection to a list to iterate over images
image_list = collection_clip.toList(collection_clip.size())

# Get collection size
size = collection_clip.size().getInfo()
print(size)

In [ ]:
#Export png - example for one image
# Get the image from the list
image = ee.Image(image_list.get(0)) #generally row number-2 from LandsatMetadata
#image.getInfo()
# 'SENSOR_ID': 'TM'  'SPACECRAFT_ID': 'LANDSAT_4',
# 'SENSOR_ID': 'TM', SPACECRAFT_ID': 'LANDSAT_5'
# 'SENSOR_ID': 'ETM',  'SPACECRAFT_ID': 'LANDSAT_7',
# 'SENSOR_ID': 'OLI_TIRS', 'SPACECRAFT_ID': 'LANDSAT_8',
# 'SENSOR_ID': 'OLI_TIRS', 'SPACECRAFT_ID': 'LANDSAT_9',

# Get image ID for filename
image_id = image.get('system:id').getInfo()
spacecraft=image.get('SPACECRAFT_ID').getInfo()

# Define output filename
#filename = os.path.join(folder_out, f'{image_id.replace("/", "_")}.png') #e.g. USDA/NAIP/DOQQ/m_4609915_sw_14_h_20170703
filename = os.path.join(folder_img, f'{image_id[-8:]}_{image_id.replace("/", "_")}.png') #add date first for better sort

# Generate thumbnail
if spacecraft == 'LANDSAT_8' or spacecraft == 'LANDSAT_9': 
    geemap.get_image_thumbnail(image,filename,vis_params=vis_params,dimensions=2000,crs='EPSG:32608') #UTM Zone 8N (EPSG:32608) 3338 Alaska Albers as a test
elif spacecraft == 'LANDSAT_7' or spacecraft == 'LANDSAT_4' or spacecraft == 'LANDSAT_5':
    geemap.get_image_thumbnail(image,filename,vis_params=vis_params45,dimensions=2000,crs='EPSG:32608') #UTM Zone 8N (EPSG:32608) 3338 Alaska Albers as a test
else:
    print('Bad spacecraft id')

print(f'Thumbnail generated for {image_id}: {filename}')
    

In [ ]:
##Export png - Loop through each image in the collection
for i in range(0,size): #for config: range(0,size,100) or range(0,10) then range(10,size) or for all use: range(size)
    try:
        # Get the image from the list
        image = ee.Image(image_list.get(i))
        
        # Get image ID for filename
        image_id = image.get('system:id').getInfo()
        spacecraft=image.get('SPACECRAFT_ID').getInfo()
        
        # Define output filename
        #filename = os.path.join(folder_out, f'{image_id.replace("/", "_")}.png') #e.g. USDA/NAIP/DOQQ/m_4609915_sw_14_h_20170703
        filename = os.path.join(folder_img, f'{image_id[-8:]}_{image_id.replace("/", "_")}.png') #add date first for better sort

        # Generate thumbnail
        if spacecraft == 'LANDSAT_8' or spacecraft == 'LANDSAT_9': 
            geemap.get_image_thumbnail(image,filename,vis_params=vis_params,dimensions=2000,crs='EPSG:32608') #UTM Zone 8N (EPSG:32608) 3338 Alaska Albers as a test
        elif spacecraft == 'LANDSAT_7' or spacecraft == 'LANDSAT_4' or spacecraft == 'LANDSAT_5':
            geemap.get_image_thumbnail(image,filename,vis_params=vis_params45,dimensions=2000,crs='EPSG:32608') #UTM Zone 8N (EPSG:32608) 3338 Alaska Albers as a test
        else:
            print('Bad spacecraft id')
        print(f'{i+1} of {size} Thumbnail generated for {image_id}: {filename}') #i+1 because python is zero-based
    except Exception as e:
        print(f'Error processing image {i}: {e}')

print('Thumbnail generation complete!')
#~15 sec/image for Hubbard - maybe area dependant? not really, seems like ~15 sec/image for all

In [ ]:
#FAILED: .map is trying to do things server-side and .getInfo is client-side. Can't mix the two. 
#ASIDE: Not sure why try/catch doesn't operate.
#test if we can do this on the collection instead of reloading each image with ee.Image
def export_thumb(image):
    try:
        # Get image ID for filename
        image_id = image.get('system:id').getInfo()
        # Define output filename
        filename = os.path.join(folder_img, f'c{image_id.replace("/", "_")}.png') #e.g. USDA/NAIP/DOQQ/m_4609915_sw_14_h_20170703
        # Generate thumbnail
        geemap.get_image_thumbnail(image,filename,vis_params=vis_params,dimensions=2000,crs='EPSG:32608') #UTM Zone 8N (EPSG:32608) 3338 Alaska Albers as a test
        print(f'Thumbnail generated for {image_id}: {filename}')
    except Exception as e:
        print(f'Error processing image {i}: {e}')
    return None #try to avoid error "User-defined methods must return a value"

# Apply export_thumb to the entire collection
#FAILS: collection_clip.map(export_thumb)